# Discover And Run Published Directory Goblins

Run this as another authorized project user after admin publication.

In [ ]:
import importlib
import os
import site
import subprocess
import sys
from pathlib import Path

package = (
    os.environ.get("GOBLIN_KING_REPOSITORY_NOTEBOOK_PACKAGE")
    or os.environ.get("GOBLIN_KING_NOTEBOOK_PACKAGE")
    or "git+https://github.com/tashabits/goblin-king.git"
)
print(f"Installing notebook helper from {package}")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--quiet",
    "--user",
    "--force-reinstall",
    "--no-deps",
    package,
])
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "goblin_king" or module_name.startswith("goblin_king."):
        del sys.modules[module_name]

import goblin_king  # noqa: E402, I001
from goblin_king.notebooks import GoblinKingNotebookClient  # noqa: E402, I001

token = os.environ.get("JUPYTERHUB_API_TOKEN") or os.environ.get("GOBLIN_KING_API_TOKEN")
if not token:
    raise RuntimeError("JUPYTERHUB_API_TOKEN or GOBLIN_KING_API_TOKEN is required")

client = GoblinKingNotebookClient(
    api_url=os.environ.get(
        "GOBLIN_KING_API_URL",
        "http://goblin-king-api.default.svc.cluster.local:8000",
    ),
    repository_url=os.environ.get("GOBLIN_KING_REPOSITORY_URL") or None,
    token=token,
    request_timeout_seconds=180,
)
print(f"Loaded goblin_king from {Path(goblin_king.__file__).resolve()}")

In [ ]:
published = client.search_directory_entries("workbook", status="published", limit=100)
[
    (
        item["entry"]["name"],
        item["entry"]["type"],
        item["entry"]["published_version"],
    )
    for item in published["items"]
]

In [ ]:
function_name = os.environ.get("GOBLIN_DIRECTORY_FUNCTION_NAME", "workbook.shared-hello")
function_run = client.run_directory_function(
    function_name,
    {"name": "Consumer"},
    progress=True,
    progress_interval_seconds=2,
)
{
    "entry": function_run["entry"],
    "version": function_run["version"],
    "job": function_run["job"],
    "run_result": function_run["run"]["result"],
}

In [ ]:
service_name = os.environ.get("GOBLIN_DIRECTORY_SERVICE_NAME", "workbook.shared-long-hello")
service = client.directory_service(service_name)
service_start = service.start(progress=True, timeout_seconds=180)
service_probe = service.probe()
service_proxy = service.proxy("/hello")
service_stop = service.stop()
{
    "service_id": service_start["service"]["id"],
    "runtime": service_start["runtime"],
    "probe": service_probe["probe"]["response"].get("json"),
    "proxy": service_proxy,
    "stopped": service_stop["notebook_service"]["runtime_status"],
}